In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
data = pd.read_csv('./dataset/HR.csv')

In [4]:
data.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,part,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14999 non-null  float64
 1   last_evaluation        14999 non-null  float64
 2   number_project         14999 non-null  int64  
 3   average_montly_hours   14999 non-null  int64  
 4   time_spend_company     14999 non-null  int64  
 5   Work_accident          14999 non-null  int64  
 6   left                   14999 non-null  int64  
 7   promotion_last_5years  14999 non-null  int64  
 8   part                   14999 non-null  object 
 9   salary                 14999 non-null  object 
dtypes: float64(2), int64(6), object(2)
memory usage: 1.1+ MB


In [6]:
data.part.unique()

array(['sales', 'accounting', 'hr', 'technical', 'support', 'management',
       'IT', 'product_mng', 'marketing', 'RandD'], dtype=object)

In [12]:
# 对于离散的字符串, 有两种处理方式
# 1. 转化成数字 2. 进行one-hot编码
data = data.join(pd.get_dummies(data.part)).join(pd.get_dummies(data.salary))

In [13]:
# 把part和salary删掉
data.drop(columns=['part', 'salary'], inplace=True)

In [14]:
data.left.value_counts()

left
0    11428
1     3571
Name: count, dtype: int64

In [15]:
11428 / (11428 + 3571)

0.7619174611640777

In [16]:
# SMOTE
Y_data = data.left.values.reshape(-1, 1)

In [17]:
Y = torch.from_numpy(Y_data).type(torch.FloatTensor)

In [18]:
Y

tensor([[1.],
        [1.],
        [1.],
        ...,
        [1.],
        [1.],
        [1.]])

In [19]:
[c for c in data.columns if c != 'left']

['satisfaction_level',
 'last_evaluation',
 'number_project',
 'average_montly_hours',
 'time_spend_company',
 'Work_accident',
 'promotion_last_5years',
 'IT',
 'RandD',
 'accounting',
 'hr',
 'management',
 'marketing',
 'product_mng',
 'sales',
 'support',
 'technical',
 'high',
 'low',
 'medium']

In [21]:
X_data = data[[c for c in data.columns if c != 'left']].values

In [30]:
X_data
X_data.shape
# X = torch.from_numpy(X_data).type(torch.FloatTensor)

(14999, 20)

In [39]:
# X = torch.from_numpy(X_data).type(torch.FloatTensor)
X_data_float = X_data.astype(np.float32)
X = torch.from_numpy(X_data_float)

In [40]:
X.shape

torch.Size([14999, 20])

In [31]:
# pytorch中最常用的一种创建模型的方式
# 子类的写法
from torch import nn

In [41]:
class HRModel(nn.Module):
    def __init__(self):
        # 先调用父类的方法
        super().__init__()
        # 定义你这个网络中会用到的东西.
        self.lin_1 = nn.Linear(20, 64)
        self.lin_2 = nn.Linear(64, 64)
        self.lin_3 = nn.Linear(64, 1)
        self.activate = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input):
        # 定义前向传播
        x = self.lin_1(input)
        x = self.activate(x)
        x = self.lin_2(x)
        x = self.activate(x)
        x = self.lin_3(x)
        x = self.sigmoid(x)
        return x

In [42]:
lr = 0.001

In [43]:
def get_model():
    model = HRModel()
    return model, torch.optim.Adam(model.parameters(), lr=lr)

In [44]:
# 定义损失函数
loss_fn = nn.BCELoss()

In [45]:
model, opt = get_model()

In [46]:
batch_size = 64
steps = len(data) // batch_size
epochs = 100

In [ ]:
# 训练过程
for epoch in range(epochs):
    for i in range(steps):
        start = i * batch_size
        end = start + batch_size
        x = X[start: end]
        y = Y[start: end]
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
print('epoch')